In [1]:
import pickle
import matplotlib.pyplot as plt
import torch
# torch.set_default_tensor_type(torch.FloatTensor)

# Load the pickle file

def load_pkl_file(path):
    with open(path, 'rb') as f:
        routing_history = pickle.load(f)
    return routing_history


In [2]:
import torch
from collections import Counter

def count_experts(expert_list):
    """
    expert_list = list of torch tensors (any shape), 
    each containing expert IDs.
    """

    all_experts = []

    for item in expert_list:
        # item is a tensor wrapped inside a list, so extract it:
        tensor_item = item[0] if isinstance(item, list) else item

        # flatten tensor to 1D
        values = tensor_item.reshape(-1).tolist()

        all_experts.extend(values)

    # count frequencies
    counter = Counter(all_experts)

    # make output dict for all 64 experts (0–63)
    result = {i: counter.get(i, 0) for i in range(64)}
    return result


# # Example usage:
# expert_counts = count_experts(routing_history.get(0))
# print(expert_counts)


In [3]:
# for i in range(0,10):
#     with_cot_path = f"OLMoE-1B-7B-0125-Instruct_with_cot_{i}_selected_experts.pkl"
#     without_cot_path = f"OLMoE-1B-7B-0125-Instruct_without_cot_{i}_selected_experts.pkl"
#     with_cot_routing_history = load_pkl_file(with_cot_path)
#     wihtout_cot_routing_history = load_pkl_file(without_cot_path)
#     for j in range(0,16):
#         if j not in [7,8,9,10,11,12,13,14,15]:
#             continue
#         print(f"For layer {j} -> \n")
#         layer_cot_data = with_cot_routing_history.get(j)
#         layer_without_cot_data = wihtout_cot_routing_history.get(j)
#         _no = min(len(layer_cot_data), len(layer_without_cot_data))
#         layer_cot_data = layer_cot_data[:_no]
#         layer_without_cot_data = layer_without_cot_data[:_no]
#         # print(len(layer_cot_data))
#         # print(len(layer_without_cot_data))
#         cot_data = list(count_experts(layer_cot_data).values())
#         wihout_cot_data = list(count_experts(layer_without_cot_data).values())
#         z = list(range(0,64))
#         plt.plot(z , cot_data,color="black")
#         plt.plot(z, wihout_cot_data, color="red")
#         plt.xlabel("X-axis")
#         plt.ylabel("Y-axis")
#         plt.title("Line Graph")
#         plt.show()
#         print("\n\n")
#     # break

#     print("\n\n\n\n")




In [4]:
expert_count_data = {i: [] for i in range(0,16)}

for i in range(0,10):
    with_cot_path = f"OLMoE-1B-7B-0125-Instruct_with_cot_{i}_selected_experts.pkl"
    without_cot_path = f"OLMoE-1B-7B-0125-Instruct_without_cot_{i}_selected_experts.pkl"
    with_cot_routing_history = load_pkl_file(with_cot_path)
    wihtout_cot_routing_history = load_pkl_file(without_cot_path)
    for j in range(0,16):
        # if j not in [7,8,9,10,11,12,13,14,15]:
        #     continue
        # print(f"For layer {j} -> \n")
        layer_cot_data = with_cot_routing_history.get(j)
        layer_without_cot_data = wihtout_cot_routing_history.get(j)
        _no = min(len(layer_cot_data), len(layer_without_cot_data))
        layer_cot_data = layer_cot_data[:_no]
        layer_without_cot_data = layer_without_cot_data[:_no]
        # print(len(layer_cot_data))
        # print(len(layer_without_cot_data))
        cot_data = list(count_experts(layer_cot_data).values())
        expert_count_data.get(j).append(cot_data)        


    #     print("\n\n")
    # # break

    # print("\n\n\n\n")

In [5]:
expert_count_data_comb = {}
import heapq

def top8_indices(values):
    # values is a list of 64 integers
    # nlargest returns (value, index) pairs for top 8 items
    top8 = heapq.nlargest(8, enumerate(values), key=lambda x: x[1])
    
    # extract only the indices
    indices = [idx for idx, val in top8]
    return indices

for i in range(0,16):
    result = [sum(x) for x in zip(*expert_count_data.get(i))]
    print(f"For layer {i}")
    top_result = top8_indices(result)
    print(top_result)
    print("\n")

    expert_count_data_comb[i] = top_result

For layer 0
[6, 58, 41, 29, 25, 33, 9, 40]


For layer 1
[11, 18, 19, 47, 42, 31, 35, 32]


For layer 2
[4, 8, 34, 45, 60, 14, 62, 32]


For layer 3
[52, 20, 61, 31, 9, 12, 38, 30]


For layer 4
[17, 49, 21, 34, 8, 52, 47, 9]


For layer 5
[0, 10, 32, 19, 6, 33, 14, 21]


For layer 6
[61, 57, 18, 5, 3, 20, 52, 1]


For layer 7
[4, 17, 25, 29, 32, 38, 35, 15]


For layer 8
[11, 54, 37, 45, 42, 16, 34, 20]


For layer 9
[5, 50, 54, 7, 51, 28, 44, 35]


For layer 10
[16, 62, 13, 56, 2, 44, 37, 17]


For layer 11
[57, 47, 43, 1, 27, 15, 44, 58]


For layer 12
[2, 63, 55, 30, 43, 11, 25, 50]


For layer 13
[27, 53, 55, 32, 46, 2, 63, 41]


For layer 14
[52, 8, 1, 13, 62, 34, 57, 63]


For layer 15
[48, 3, 17, 39, 44, 22, 59, 51]




In [4]:
import torch
from collections import defaultdict

def count_experts_and_weights(expert_ids, expert_weights):
    """
    expert_ids: tensor of shape (N, K)   -> expert indices
    expert_weights: tensor of shape (N, K) -> expert weights
    """

    count_dict = defaultdict(int)
    weight_sum_dict = defaultdict(float)

    N, K = expert_ids.shape

    # Loop through all tokens and experts
    for i in range(N):
        for j in range(K):
            expert = int(expert_ids[i, j].item())
            weight = float(expert_weights[i, j].item())
            
            count_dict[expert] += 1
            weight_sum_dict[expert] += weight

    # Convert defaultdict to normal dicts
    count_dict = dict(count_dict)
    weight_sum_dict = dict(weight_sum_dict)

    # (Optional) compute average weight
    avg_weight_dict = {
        expert: weight_sum_dict[expert] / count_dict[expert]
        for expert in count_dict
    }

    return count_dict, weight_sum_dict, avg_weight_dict



def merge_expert_weights(dict_list):
    final = {}

    for d in dict_list:
        for expert, weight in d.items():
            if expert not in final:
                final[expert] = weight
            else:
                final[expert] += weight

    return final

In [5]:
data = {i:[] for i in range(0,16)}
for i in range(0,70):
    router_path = f"output/router/OLMoE-1B-7B-0125-Instruct_with_cot_{i}_selected_experts.pkl"
    wieght_path = f"output/weight_router/OLMoE-1B-7B-0125-Instruct_with_cot_{i}_selected_experts.pkl"
    routing_history = load_pkl_file(router_path)
    weight_history = load_pkl_file(wieght_path)
    for j in range(0,16):
        # if j not in [7,8,9,10,11,12,13,14,15]:
        #     continue
        # print(f"For layer {j} -> \n")
        router_cot_data = routing_history.get(j)
        weight_cot_data = weight_history.get(j)
        # print(router_cot_data)
        # print(weight_cot_data)
        for k in range(0,len(router_cot_data)):

            count_dict, weight_sum_dict, avg_weight_dict = count_experts_and_weights(
                router_cot_data[k][0], weight_cot_data[k][0]
            )
            data[j].append(weight_sum_dict)


    # print("\n\n\n\n")




In [6]:
def extract_top_experts(top):
    experts = [e for e, _ in top]
    weights = [w for _, w in top]
    # normalize
    total = sum(weights)
    normalized_weights = [round(w / total, 4) for w in weights]

    return experts, normalized_weights

# routing_history = {i:[] for i in range(0,16)}
# routing_weight = {i:[] for i in range(0,19)}
routing_history = {}
routing_weight = {}

for j in range(0,16):
    # print(f"For layer {j} -> \n")
    final_weights = merge_expert_weights(data.get(j))
    sorted_items = sorted(final_weights.items(), key=lambda x: x[1], reverse=True)
    # Top-8
    top8 = sorted_items[:8]
    experts, weights = extract_top_experts(top8)
    # routing_history[j].append(experts)
    # routing_weight[j].append(weights)
    routing_history[j] = experts
    routing_weight[j] = weights


In [7]:
routing_history, routing_weight

({0: [58, 6, 41, 29, 25, 33, 10, 38],
  1: [11, 18, 47, 19, 48, 42, 21, 53],
  2: [34, 8, 4, 32, 30, 14, 26, 62],
  3: [52, 31, 20, 61, 12, 9, 29, 38],
  4: [34, 21, 17, 52, 8, 49, 14, 6],
  5: [0, 10, 19, 32, 6, 33, 14, 21],
  6: [61, 57, 18, 20, 5, 3, 1, 62],
  7: [4, 29, 32, 17, 25, 38, 35, 61],
  8: [11, 54, 37, 45, 16, 34, 62, 20],
  9: [5, 7, 50, 54, 51, 1, 28, 44],
  10: [13, 62, 5, 16, 56, 2, 11, 37],
  11: [57, 47, 43, 15, 44, 1, 27, 56],
  12: [2, 63, 55, 30, 14, 41, 43, 46],
  13: [27, 32, 55, 53, 46, 2, 54, 41],
  14: [52, 1, 34, 57, 63, 8, 9, 60],
  15: [48, 3, 44, 17, 53, 22, 39, 20]},
 {0: [0.1858, 0.1739, 0.1467, 0.1394, 0.1196, 0.0824, 0.0774, 0.0749],
  1: [0.2492, 0.1339, 0.1161, 0.1078, 0.1045, 0.0993, 0.0959, 0.0932],
  2: [0.212, 0.1847, 0.1582, 0.1001, 0.0908, 0.0907, 0.0821, 0.0813],
  3: [0.3168, 0.1267, 0.1186, 0.1133, 0.0865, 0.0825, 0.0809, 0.0747],
  4: [0.1587, 0.1551, 0.1377, 0.132, 0.1235, 0.1225, 0.0877, 0.0828],
  5: [0.2325, 0.1957, 0.1432, 0.1308, 0.

In [24]:
experts

[48, 44, 3, 22, 39, 17, 53, 20]